# Planner + Executor: Runtime Tool Discovery with AWS Agent Registry

## Overview

When building AI agents that need to discover and use tools at runtime, loading every
available tool into the LLM context upfront is expensive, slow, and wasteful. As your
tool catalog grows — across order management, payments, shipping, notifications, and
more — the token cost and latency of stuffing all tool schemas into every request
becomes a real bottleneck.

This notebook demonstrates the **Planner + Executor pattern**, a two-phase approach to
runtime tool discovery that solves this problem:

1. **Planner Agent** — receives a task, searches the AWS Agent Registry to discover
   which tools are relevant, and outputs a minimal Tool Plan. It never executes
   business logic — it only identifies what's needed.

2. **Executor Agent** — loads only the tools specified in the Plan, creates live
   connections to MCP servers and A2A agents, and executes the task step by step.

This pattern delivers three key benefits:

- **Runtime discovery** — agents find the right tools dynamically from the Registry
  instead of hardcoding them, so new tools become available without redeploying agents
- **Token optimization** — the Executor's context contains only the tools it needs
  (e.g., 2-3 out of 12), significantly reducing input tokens and cost compared to
  loading everything upfront
- **Faster execution** — smaller context means faster LLM inference and fewer
  irrelevant tool calls

In this notebook you will be using **real deployed services** — 9 MCP tools on AgentCore Gateway and
3 A2A agents on AgentCore Runtime — to run 3 end-to-end e-commerce scenarios. You will perform runtime discovery of the required MCP servers and A2A agents for each scenario using Planner agent and then use Executor agent to execute the scenario based on the plan and tools provided by the Planner agent. You will also analyze the token and cost savings that comes with runtime tool discovery and execution.

## Deploying MCP tools to Amazon AgentCore Gateway and A2A agents to Amazon AgentCore Runtime

Run these notebooks first to deploy the tools and agents:
1. `00-deploy-sample-mcp-tools.ipynb` — deploys 9 MCP tools on AgentCore Gateway
2. `00-deploy-sample-a2a-agents.ipynb` — deploys 3 A2A agents on AgentCore Runtime

Both notebooks save config files (`utils/mcp_tools_config.json`, `util/sa2a_agents_config.json`)
that this notebook loads automatically.

**These MCP servers and A2A agents will be dpeloyed to Amazon AgentCore once above notebooks are executed**

| Tool | Protocol | Purpose |
|---|---|---|
| `order_lookup_tool` | MCP | Look up order details and list orders by customer |
| `order_update_tool` | MCP | Update order status or shipping address |
| `order_cancel_tool` | MCP | Cancel an order |
| `email_send_tool` | MCP | Send transactional emails to customers |
| `email_template_tool` | MCP | Manage reusable email templates |
| `sms_notify_tool` | MCP | Send SMS notifications |
| `payment_status_tool` | MCP | Look up payment status for an order |
| `inventory_check_tool` | MCP | Check available stock for one or more SKUs |
| `shipping_track_tool` | MCP | Track shipments and get delivery estimates |
| `returns_processing_tool` | MCP (remote) | Process product returns and generate return labels |
| `loyalty_rewards_tool` | MCP (remote) | Manage loyalty points and redeem rewards |
| `payment_refund_tool` | A2A | Issue refunds with multi-step validation |
| `inventory_reserve_tool` | A2A | Reserve inventory with rollback support |
| `shipping_update_tool` | A2A | Create shipments with carrier selection and status updates |

## Architecture

![Planner + Executor architecture](images/planner-executor-architecture.png)

## Sections
1. Setup
2. Obtain OAuth2 Token
3. Create Registry & Register All 12 Tools
4. Build Planner Agent
5. Build Executor Agent
6. Run Planner + Executor Locally (3 scenarios)
7. Token Comparison
8. Deploy to AgentCore Runtime
9. Invoke Deployed Agent
10. Cleanup

---
## Step 1: Setup

### Prerequisites
- AWS account with IAM credentials configured
- Python 3.10+
- boto3 >= 1.42.87
- IAM user or role with the permissions below (replace ACCOUNT_ID and REGION as needed)

<details>
<summary>Required IAM policy (click to expand)</summary>

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowCreateRegistry",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:CreateRegistry"],
            "Resource": ["arn:aws:bedrock-agentcore:us-west-2:ACCOUNT_ID:*"]
        },
        {
            "Sid": "AllowGetUpdateDeleteRegistry",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistry",
                "bedrock-agentcore:DeleteRegistry"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:us-west-2:ACCOUNT_ID:registry/*"]
        },
        {
            "Sid": "AllowCreateAndListRecords",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateRegistryRecord",
                "bedrock-agentcore:SearchRegistryRecords"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:us-west-2:ACCOUNT_ID:registry/*"]
        },
        {
            "Sid": "AllowRecordOperations",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistryRecord",
                "bedrock-agentcore:DeleteRegistryRecord",
                "bedrock-agentcore:SubmitRegistryRecordForApproval"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:us-west-2:ACCOUNT_ID:registry/*/record/*"]
        }
    ]
}
```

</details>

**Note:** This notebook uses `bedrock-agentcore`, `bedrock-agentcore-starter-toolkit`, `strands-agents`, and `matplotlib`. These are installed automatically via `requirements.txt`.

### Auth0 Setup (Optional)

To use OAuth-based registry search instead of IAM, set up Auth0:

1. Create an Auth0 Account & Tenant
- Sign up at [auth0.com](https://auth0.com)
- Create a new tenant — note your **Auth0 Domain** (e.g., `your-tenant.us.auth0.com`)

2. Register an API (Resource Server)
- Navigate to **Applications → APIs** in the Auth0 Dashboard
- Click **+ Create API**
- Set the **Identifier (Audience)** to: `https://bedrock-agentcore.us-west-2.amazonaws.com`
- Select **RS256** signing algorithm

3. Configure .env
- Copy `03-Consumer/.env.example` to `03-Consumer/.env`
- Fill in `AUTH0_DOMAIN`, `AUTH0_CLIENT_ID`, `AUTH0_CLIENT_SECRET`
- Leave Auth0 fields empty to fall back to IAM auth


#### Install the dependencies

In [ ]:
!pip install -r requirements.txt

#### Connect to your AWS environment

In [ ]:
import boto3, json, time, os, sys, pathlib, math, uuid
from dotenv import load_dotenv
from datetime import datetime
from strands import Agent, tool
from strands.models import BedrockModel

# ── Load .env for Auth0 config (optional) ────────────────────────────────
load_dotenv(pathlib.Path.cwd() / ".env")

# Auth0 OAuth config (leave empty to use IAM auth for registry search)
AUTH0_DOMAIN = os.getenv("AUTH0_DOMAIN", "")
AUTH0_CLIENT_ID = os.getenv("AUTH0_CLIENT_ID", "")
AUTH0_CLIENT_SECRET = os.getenv("AUTH0_CLIENT_SECRET", "")
AUTH0_AUDIENCE = os.getenv("AUTH0_AUDIENCE", "")
USE_AUTH0 = bool(AUTH0_DOMAIN and AUTH0_CLIENT_ID and AUTH0_CLIENT_SECRET)

print(f"Auth mode: {'Auth0 OAuth' if USE_AUTH0 else 'IAM (default)'}")

# ── Config ─────────────────────────────────────────────────────────────────

# Set AWS credentials if not using Amazon SageMaker notebook
#AWS_PROFILE = "configured-aws-profile"

AWS_REGION  = "us-west-2"
MODEL_ID    = "us.anthropic.claude-sonnet-4-20250514-v1:0"

session          = boto3.Session(region_name=AWS_REGION)

cp_client        = session.client("bedrock-agentcore-control")
dp_client        = session.client("bedrock-agentcore")


iam_client        = session.client("iam")
ACCOUNT_ID        = session.client("sts").get_caller_identity()["Account"]

def pp(data):
    print(json.dumps(data, indent=2, default=str))

def estimate_tokens(text: str) -> int:
    return math.ceil(len(text) / 4)


print(f"Account     : {ACCOUNT_ID}")
print(f"Region      : {AWS_REGION}")

### Load deployment configs for MCP tools and A2A agents from prior deployment notebooks

In [ ]:
mcp_config = json.loads(pathlib.Path("utils/mcp_tools_config.json").read_text())
a2a_config = json.loads(pathlib.Path("utils/a2a_agents_config.json").read_text())

print(f"MCP Gateway : {mcp_config['gateway_url']}")
print(f"A2A Agents  : {list(a2a_config['agents'].keys())}")
print("Clients ready.")

---
## Step 2: Obtain OAuth2 Tokens

Get a Cognito access token for the MCP Gateway (always needed).
If Auth0 is configured, also obtain an Auth0 token for OAuth-based registry search.


In [ ]:
import base64, requests

# ── MCP Gateway token (Cognito — always needed for Gateway tools) ─────────
credentials = base64.b64encode(
    f"{mcp_config['client_id']}:{mcp_config['client_secret']}".encode()
).decode()

token_resp = requests.post(
    f"https://{mcp_config['cognito_domain']}/oauth2/token",
    headers={"Authorization": f"Basic {credentials}",
             "Content-Type": "application/x-www-form-urlencoded"},
    data={"grant_type": "client_credentials",
          "scope": f"{mcp_config['resource_server_id']}/read {mcp_config['resource_server_id']}/write"},
    timeout=30
)
token_resp.raise_for_status()
access_token = token_resp.json()["access_token"]
print(f"MCP Gateway token obtained ({len(access_token)} chars)")

# ── Auth0 token for Registry search (optional) ───────────────────────────
auth0_token = None
if USE_AUTH0:
    auth0_resp = requests.post(
        f"https://{AUTH0_DOMAIN}/oauth/token",
        json={
            "grant_type": "client_credentials",
            "client_id": AUTH0_CLIENT_ID,
            "client_secret": AUTH0_CLIENT_SECRET,
            "audience": AUTH0_AUDIENCE,
        },
        timeout=30,
    )
    auth0_resp.raise_for_status()
    auth0_token = auth0_resp.json()["access_token"]
    print(f"Auth0 token obtained ({len(auth0_token)} chars)")
else:
    print("Auth0 not configured — using IAM for registry search")


---
## Step 3: Create Registry & Register All 12 Tools

Populate the Registry with all 12 deployed tools and agents so the Planner can discover them at runtime.
Each MCP record includes the Gateway URL for live tool execution. Each A2A record includes the agent card URL for direct agent invocation.

### Create Registry

In [ ]:
from utils.registry_records import create_and_approve_all_records

# Create registry — Auth0 OAuth or IAM based on config
if USE_AUTH0:
    discovery_url = f"https://{AUTH0_DOMAIN}/.well-known/openid-configuration"
    reg = cp_client.create_registry(
        name="plannerExecutorLiveDemo",
        description="Registry for live Planner+Executor demo with Auth0 OAuth",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": discovery_url,
                "allowedAudience": [AUTH0_AUDIENCE],
            }
        },
    )
else:
    reg = cp_client.create_registry(
        name="plannerExecutorLiveDemo",
        description="Registry for live Planner+Executor demo",
        approvalConfiguration={"autoApproval": False}
    )

REGISTRY_ARN = reg["registryArn"]
REGISTRY_ID  = REGISTRY_ARN.split("/")[-1]
print(f"Registry created : {REGISTRY_ID}")
print(f"Registry ARN     : {REGISTRY_ARN}")

# Wait for READY
while True:
    r = cp_client.get_registry(registryId=REGISTRY_ID)
    if r["status"] == "READY":
        print("Registry status  : READY"); break
    if r["status"] == "CREATE_FAILED":
        print(f"Registry FAILED: {r.get('statusReason', 'unknown')}"); break
    print(f"Registry status  : {r['status']} — waiting...")
    time.sleep(5)

# Add MCP URL to allowedAudience for Auth0 registries
if USE_AUTH0:
    mcp_url = f"{AUTH0_AUDIENCE}/registry/{REGISTRY_ID}/mcp"
    registry_info = cp_client.get_registry(registryId=REGISTRY_ID)
    jwt_config = registry_info["authorizerConfiguration"]["customJWTAuthorizer"]
    audience = list(set(jwt_config.get("allowedAudience", []) + [mcp_url]))
    cp_client.update_registry(
        registryId=REGISTRY_ID,
        authorizerConfiguration={
            "optionalValue": {
                "customJWTAuthorizer": {
                    "discoveryUrl": jwt_config["discoveryUrl"],
                    "allowedAudience": audience,
                }
            }
        },
    )
    # Wait for update
    while True:
        status = cp_client.get_registry(registryId=REGISTRY_ID)["status"]
        if status != "UPDATING": break
        time.sleep(2)
    print(f"Added MCP URL to allowedAudience: {mcp_url}")


### Build registry record definitions from saved configs

In [ ]:
# ── Build registry record definitions from saved configs ───────────────────
gateway_url = mcp_config["gateway_url"]

# 9 MCP tool records
TOOL_RECORDS = []
for t in mcp_config["tools"]:
    TOOL_RECORDS.append({
        "name": t["name"],
        "description": t["description"],
        "descriptorType": "MCP",
        "descriptors": {
            "mcp": {
                "server": {
                    "inlineContent": json.dumps({
                        "name": f"gateway-mcp-server/{t['name']}",
                        "description": t["description"],
                        "version": "1.0.0",
                        "websiteUrl": gateway_url
                    })
                },
                "tools": {
                    "inlineContent": json.dumps({
                        "tools": t.get("tools_full", [{"name": tn, "description": tn, "inputSchema": {"type": "object"}} for tn in t["tool_names"]])
                    })
                }
            }
        }
    })

# 3 A2A agent records
from urllib.parse import quote
for key, agent in a2a_config["agents"].items():
    escaped_arn = quote(agent["agent_arn"], safe="")
    agent_url = f"{AUTH0_AUDIENCE}/runtimes/{escaped_arn}/invocations/"
    agent_card = {
        "protocolVersion": "0.3",
        "name": agent["name"],
        "description": agent["description"],
        "version": "1.0.0",
        "url": agent_url,
        "capabilities": {"streaming": True},
        "skills": [{"id": s, "name": s, "description": s, "tags": [s]} for s in agent.get("skills", [])],
        "defaultInputModes": ["text/plain"],
        "defaultOutputModes": ["text/plain"],
    }
    TOOL_RECORDS.append({
        "name": f"{key}_tool",
        "description": agent["description"],
        "descriptorType": "A2A",
        "descriptors": {
            "a2a": {
                "agentCard": {
                    "schemaVersion": "0.3",
                    "inlineContent": json.dumps(agent_card)
                }
            }
        }
    })

print(f"Defined {len(TOOL_RECORDS)} records: {len(mcp_config['tools'])} MCP + {len(a2a_config['agents'])} A2A")

### Insert registry record definitions into AWS registry

In [ ]:
# Create and approve all records (create → submit → approve)
from utils.registry_records import create_and_approve_all_records
#import utils.registry_records


record_ids = create_and_approve_all_records(cp_client, REGISTRY_ID, TOOL_RECORDS)

print(f"\nAll {len(record_ids)} records approved.")
print("Waiting for search index to update (eventual consistency)...")
time.sleep(15)
print("Registry ready for search.")

---
## Step 4: Build the Planner Agent

The Planner Agent has a single responsibility: given a task, search the Registry to discover which tools are needed and output a structured Tool Plan as JSON. It never executes business logic or calls tools directly — it only identifies the minimal set of tools required.

In [ ]:
@tool
def search_registry(query: str) -> str:
    """Search the AWS Agent Registry to discover available tools and agents.

    Use this tool to find tools that match a capability needed for the task.
    Search once per distinct capability (e.g., search 'order update' separately
    from 'email notification').

    Args:
        query: Natural language description of the capability needed.

    Returns:
        JSON list of matching registry records with available tool names.
    """
    try:
        if USE_AUTH0 and auth0_token:
            # OAuth bearer token search via HTTP
            import requests as _req
            resp = _req.post(
                f"{AUTH0_AUDIENCE}/registry-records/search",
                headers={"Content-Type": "application/json",
                         "Authorization": f"Bearer {auth0_token}"},
                json={"searchQuery": query, "registryIds": [REGISTRY_ARN], "maxResults": 3},
                timeout=120,
            )
            resp.raise_for_status()
            records = resp.json().get("registryRecords", [])
        else:
            # IAM auth search via boto3
            results = dp_client.search_registry_records(
                registryIds=[REGISTRY_ARN],
                searchQuery=query,
                maxResults=3
            )
            records = results.get("registryRecords", [])
        if not records:
            return json.dumps({"message": "No matching tools found", "query": query})
        out = []
        for rec in records:
            entry = {
                "record_id":   rec["recordId"],
                "name":        rec["name"],
                "description": rec.get("description", ""),
                "descriptorType":    rec.get("descriptorType", "MCP"),
            }
            if rec.get("descriptorType") == "A2A":
                ac = (rec.get("descriptors", {}).get("a2a", {})
                         .get("agentCard", {}).get("inlineContent", "{}"))
                card = json.loads(ac)
                entry["available_tools"] = [s.get("id", s.get("name", "")) for s in card.get("skills", [])]
            else:
                tc = (rec.get("descriptors", {}).get("mcp", {})
                         .get("tools", {}).get("inlineContent", "{}"))
                entry["available_tools"] = [t["name"] for t in json.loads(tc).get("tools", [])]
            out.append(entry)
        return json.dumps({"results_count": len(out), "records": out}, indent=2)
    except Exception as ex:
        return json.dumps({"error": str(ex)})


PLANNER_SYSTEM_PROMPT = """You are a Planner agent. Your ONLY job is to analyse a task and
identify the minimal set of registry tools needed to complete it.

Rules:
- Use search_registry to find relevant tools. Search once per distinct capability.
- Do NOT execute any business logic or make up data.
- Return ONLY a JSON object in exactly this format — no prose, no markdown fences:

{
  "task": "<original task>",
  "steps": [
    {
      "step": 1,
      "description": "<what this step does>",
      "record_id": "<registry record ID>",
      "record_name": "<registry record name>",
      "descriptorType": "MCP or A2A",
      "tool_name": "<specific tool name from available_tools>"
    }
  ],
  "selected_record_ids": ["<id1>", "<id2>"]
}

selected_record_ids must contain only the unique record IDs used across all steps."""

planner_model = BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION)
planner_agent = Agent(
    model=planner_model,
    tools=[search_registry],
    system_prompt=PLANNER_SYSTEM_PROMPT
)
print("Planner agent created.")

---
## Step 5: Build the Executor Agent

The Executor Agent receives the Tool Plan from the Planner and dynamically builds live connections to only the planned tools.
For MCP tools, it connects to the Gateway via streamable HTTP with OAuth2 authentication.
For A2A agents, it invokes them via the AgentCore Runtime data plane.
No mocks — every tool call hits a real deployed service.

In [ ]:
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from urllib.parse import unquote
from utils.tool_builder import parse_mcp_metadata, parse_a2a_metadata, create_a2a_tool

def build_executor(tool_plan: dict) -> tuple:
    """Build an Executor agent with live tool connections from the Tool Plan."""
    selected_ids   = tool_plan.get("selected_record_ids", [])
    fetched_schemas = {}
    dynamic_tools   = []
    seen_mcp_urls   = set()

    print(f"Fetching {len(selected_ids)} record(s) from Registry...")
    for rid in selected_ids:
        record = cp_client.get_registry_record(
            registryId=REGISTRY_ID, recordId=rid)
        record_name     = record["name"]
        descriptor_type = record.get("descriptorType", "MCP")
        fetched_schemas[rid] = record

        if descriptor_type == "MCP":
            meta = parse_mcp_metadata(record)
            print(f"  [MCP]  {record_name}  tools: {meta['tool_names']}")

            if meta["url"] and meta["url"] not in seen_mcp_urls:
                headers = {"Authorization": f"Bearer {access_token}"}
                mcp_client = MCPClient(
                    lambda u=meta["url"], h=headers: streamablehttp_client(u, headers=h))
                dynamic_tools.append(mcp_client)
                seen_mcp_urls.add(meta["url"])

        elif descriptor_type == "A2A":
            meta = parse_a2a_metadata(record)
            print(f"  [A2A]  {record_name}  skills: {meta['skills']}")

            if meta["url"]:
                # Extract agent ARN directly from the registry record URL
                agent_arn = None
                parts = meta["url"].split("/runtimes/")
                if len(parts) > 1:
                    agent_arn = unquote(parts[1].split("/invocations")[0])

                if agent_arn:
                    dynamic_tools.append(
                        create_a2a_tool(
                            record_name,
                            record.get("description", ""),
                            agent_arn,
                            meta["skills"],
                            dp_client))

    executor_model = BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION)
    executor_agent = Agent(
        model=executor_model,
        tools=dynamic_tools,
        system_prompt=(
            "You are an Executor agent. Execute this Tool Plan step by step "
            "using the provided tools. After all steps complete, provide a "
            "concise summary of what was accomplished.\n\n"
            f"Tool Plan:\n{json.dumps(tool_plan, indent=2)}"
        )
    )
    return executor_agent, fetched_schemas

print("Executor builder ready (live connections).")

---
## Step 6: Run Planner + Executor Locally

Run the full Planner → Executor pipeline against three real e-commerce scenarios. Each scenario exercises a different combination of MCP tools and A2A agents, demonstrating how the pattern adapts to varying task requirements.

### Orchestration Helper

In [ ]:
def run_planner_executor(task: str):
    """Run the full Planner → Executor pipeline for a task."""
    print("=" * 65)
    print("PHASE 1 — PLANNER")
    print("=" * 65)
    print(f"Task: {task}\n")

    planner_response = planner_agent(task)
    planner_raw = planner_response.message["content"][0]["text"]

    clean = planner_raw.strip()
    if "```" in clean:
        # Extract content between first pair of ``` fences
        parts = clean.split("```")
        for part in parts[1:]:
            p = part.strip()
            if p.startswith("json"):
                p = p[4:].strip()
            if p.startswith("{"):
                clean = p
                break
    # Fallback: find JSON object in the raw text
    import re
    if not clean.startswith("{"):
        match = re.search(r'\{.*\}', clean, re.DOTALL)
        if match:
            clean = match.group()
    tool_plan = json.loads(clean.strip())

    print("\nTool Plan:")
    pp(tool_plan)
    print(f"\nPlanner selected {len(tool_plan['selected_record_ids'])} record(s) "
          f"out of {len(TOOL_RECORDS)} in the registry.")

    print("\n" + "=" * 65)
    print("PHASE 2 — EXECUTOR")
    print("=" * 65)

    executor_agent, fetched_schemas = build_executor(tool_plan)
    print(f"\nExecuting {len(tool_plan['steps'])} step(s)...\n")

    executor_response = executor_agent(
        f"Execute the following task using the Tool Plan provided: {task}"
    )

    print("\n" + "=" * 65)
    print("FINAL RESULT")
    print("=" * 65)
    result_text = executor_response.message["content"][0]["text"]
    print(result_text)

    return tool_plan, fetched_schemas, result_text

### Scenario 1: Cancel Order + Full Refund

Customer wants to cancel order ORD-1002 and get a full refund. This should exercise
the order cancellation MCP tool and the payment refund A2A agent.

In [ ]:
plan1, schemas1, result1 = run_planner_executor(
    "Cancel order ORD-1002 and issue a full refund to the customer."
)

### Scenario 2: Inventory Reserve + Stock Check

Reserve inventory for a new order and verify stock levels. Exercises the inventory
reserve A2A agent and the inventory check MCP tool.

In [ ]:
plan2, schemas2, result2 = run_planner_executor(
    "Reserve 5 units of WIDGET-42 for order ORD-1004, then check the current inventory level for WIDGET-42."
)

### Scenario 3: Ship Order + Email Notification

Create a shipment for an order and send a confirmation email. Exercises the shipping
update A2A agent, order update MCP tool, and email send MCP tool.

In [ ]:
plan3, schemas3, result3 = run_planner_executor(
    "Ship order ORD-1001: assign a carrier for a 2kg package to WA, create the shipment, "
    "update the order status to SHIPPED, and send a shipping confirmation email to the customer."
)

---
## Step 7: Token Comparison

Quantify the token savings of the Planner+Executor approach. The baseline loads all 12 tool schemas into the LLM context upfront. The optimised approach loads only the 2-3 tools the Planner selected, plus a small overhead for the Planner's own Registry search calls.

In [ ]:
# ── Token Comparison (all 3 scenarios) ──────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

scenarios = [
    ("Scenario 1:\nCancel + Refund", plan1, schemas1),
    ("Scenario 2:\nReserve + Check", plan2, schemas2),
    ("Scenario 3:\nShip + Notify",   plan3, schemas3),
]

tokens_all = estimate_tokens(json.dumps(TOOL_RECORDS))

planned_tokens  = []
planner_tokens  = []
for label, plan, schemas in scenarios:
    planned_text = json.dumps([
        {"name": schemas[rid]["name"],
         "description": schemas[rid].get("description", ""),
         "descriptors": schemas[rid].get("descriptors", {})}
        for rid in plan["selected_record_ids"]
    ])
    planner_text = PLANNER_SYSTEM_PROMPT + json.dumps(plan)
    planned_tokens.append(estimate_tokens(planned_text))
    planner_tokens.append(estimate_tokens(planner_text))

optimised_tokens = [p + o for p, o in zip(planned_tokens, planner_tokens)]
reduction_pcts   = [((tokens_all - opt) / tokens_all * 100) for opt in optimised_tokens]

# ── Grouped Bar Chart ────────────────────────────────────────────────────
labels = [s[0] for s in scenarios]
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("Planner+Executor vs. Load-All-Tools Upfront", fontsize=14, fontweight="bold")

# Baseline bars
ax.bar(x - width/2, [tokens_all] * len(scenarios), width,
       color="#e74c3c", label="Baseline (all 12 tools)")

# Optimised bars (stacked: planned + planner overhead)
ax.bar(x + width/2, planned_tokens, width,
       color="#2ecc71", label="Executor (planned tools)")
ax.bar(x + width/2, planner_tokens, width,
       bottom=planned_tokens, color="#f39c12", label="Planner overhead")

# Labels
for i in range(len(scenarios)):
    ax.text(x[i] - width/2, tokens_all + tokens_all * 0.02,
            f"{tokens_all:,}", ha="center", va="bottom", fontsize=8)
    ax.text(x[i] + width/2, optimised_tokens[i] + tokens_all * 0.02,
            f"{optimised_tokens[i]:,}\n({reduction_pcts[i]:.0f}% less)",
            ha="center", va="bottom", fontsize=8, color="#2e7d32")

ax.set_title("Input Tokens per Scenario")
ax.set_ylabel("Tokens (estimated)")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

for i, (label, plan, _) in enumerate(scenarios):
    print(f"{label.replace(chr(10), ' ')} — "
          f"Tools: {len(plan['selected_record_ids'])}/{len(TOOL_RECORDS)}  |  "
          f"Tokens: {optimised_tokens[i]:,} vs {tokens_all:,}  |  "
          f"Reduction: {reduction_pcts[i]:.0f}%")

---
## Step 8: Cleanup

Clean up the Registry created by this notebook. The MCP tools and A2A agents deployed by the prerequisite notebooks are left running — manage them from their respective notebooks.

In [ ]:
# 1. Delete registry records + registry
print("Cleaning up Registry...")
for rid in record_ids:
    try:
        cp_client.delete_registry_record(registryId=REGISTRY_ID, recordId=rid)
        print(f"  Deleted record: {rid}")
    except Exception as e:
        print(f"  Record {rid}: {e}")
time.sleep(5)
try:
    cp_client.delete_registry(registryId=REGISTRY_ID)
    print(f"  Deleted registry: {REGISTRY_ID}")
except Exception as e:
    print(f"  Registry: {e}")

# 2. Remove local files
print("\nCleaning up local files...")
for f in ["planner_executor_live_comparison.png"]:
    p = pathlib.Path(f)
    if p.exists(): p.unlink(); print(f"  Removed: {f}")

print("\nCleanup complete! MCP tools and A2A agents are still running (managed by their own notebooks).")

---
## What We Demonstrated

This notebook showed how the Planner + Executor pattern enables efficient runtime tool discovery:

1. Registered 12 tools (9 MCP + 3 A2A) in a single AWS Agent Registry
2. The Planner searched the Registry and selected only the tools needed for each task
3. The Executor built live connections to real MCP Gateway targets and A2A agents on AgentCore Runtime
4. Ran 3 end-to-end e-commerce scenarios — order cancellation with refund, inventory reservation, and shipment creation with email notification
5. Measured token and cost savings vs. loading all tools upfront

The key takeaway: as your tool catalog grows, this pattern keeps agent execution fast and cost-efficient by loading only what's needed, while the Registry ensures new tools are discoverable without redeploying agents.